# Final RF-25 ET maps — fine and native MODIS products

Visualization only. Inputs are read from the repository-local `outputs/` tree.

- Fine products: final 20 m RF-25 ET rasters after weighted-AOA masking and exact-overlap reconciliation.
- Coarse products: native-grid MOD16A2GF v6.1 ET rasters used by the reconciliation.
- AOA/DI/LPD are stored as bands in the fine product and are not interpreted as independent ET validation.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import rasterio
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter, MaxNLocator
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from rasterio.warp import (
    Resampling,
    calculate_default_transform,
    reproject,
    transform,
    transform_geom,
)
from shapely.geometry import shape


FINAL_DATES = ["2020-03-13", "2021-11-25", "2022-03-30"]
DISPLAY_VMIN = 0.0
DISPLAY_VMAX = 100.0

STATION_LABEL_OFFSETS = {
    "ST01": (6, 8),
    "ST02": (-42, -15),
    "ST03": (7, 8),
    "ST04": (7, -12),
    "ST05": (7, 8),
}


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (
            (candidate / "src" / "et_downscaling").exists()
            and (candidate / "data").exists()
        ):
            return candidate
    raise FileNotFoundError("Repository root could not be located.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
WORKSPACE = REPO_ROOT / "outputs"
FINAL_DIR = WORKSPACE / "current" / "rasters"
MODIS_DIR = WORKSPACE / "current" / "rasters_modis"
OUTPUT_DIR = WORKSPACE / "current" / "figures" / "final_maps"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BASIN_PATH = REPO_ROOT / "data" / "boundaries" / "fundacion_basin.geojson"
STATIONS_PATH = REPO_ROOT / "data" / "stations" / "fundacion_stations.geojson"

for path in [FINAL_DIR, MODIS_DIR, BASIN_PATH, STATIONS_PATH]:
    if not path.exists():
        raise FileNotFoundError(path)

print("Repository:", REPO_ROOT)
print("Fine rasters:", FINAL_DIR)
print("Native MODIS rasters:", MODIS_DIR)
print("Map output:", OUTPUT_DIR)

In [ ]:
def load_geojson_features(path: Path, target_crs: str = "EPSG:4326"):
    data = json.loads(path.read_text(encoding="utf-8"))
    source_crs = "EPSG:4326"

    crs_info = data.get("crs", {})
    crs_name = (
        crs_info.get("properties", {}).get("name")
        if isinstance(crs_info, dict)
        else None
    )
    if crs_name:
        source_crs = crs_name

    features = []
    for feature in data["features"]:
        geometry = feature.get("geometry")
        if geometry is None:
            continue

        if source_crs != target_crs:
            geometry = transform_geom(
                source_crs,
                target_crs,
                geometry,
                precision=12,
            )

        features.append(
            {
                "geometry": shape(geometry),
                "properties": feature.get("properties", {}),
            }
        )
    return features


def iter_polygon_exteriors(geometry):
    if geometry.geom_type == "Polygon":
        yield geometry.exterior
    elif geometry.geom_type == "MultiPolygon":
        for polygon in geometry.geoms:
            yield polygon.exterior


def feature_bounds(features):
    bounds = np.asarray(
        [feature["geometry"].bounds for feature in features],
        dtype=float,
    )
    return (
        float(bounds[:, 0].min()),
        float(bounds[:, 1].min()),
        float(bounds[:, 2].max()),
        float(bounds[:, 3].max()),
    )


basin_features = load_geojson_features(BASIN_PATH)
station_features = load_geojson_features(STATIONS_PATH)

basin_bounds = feature_bounds(basin_features)
station_bounds = feature_bounds(station_features)

MAP_BOUNDS = (
    min(basin_bounds[0], station_bounds[0]),
    min(basin_bounds[1], station_bounds[1]),
    max(basin_bounds[2], station_bounds[2]),
    max(basin_bounds[3], station_bounds[3]),
)

print("Stations:", len(station_features))
print("Map bounds:", MAP_BOUNDS)

In [ ]:
def reproject_raster_for_display(
    raster_path: Path,
    target_crs: str = "EPSG:4326",
):
    with rasterio.open(raster_path) as src:
        dst_transform, width, height = calculate_default_transform(
            src.crs,
            target_crs,
            src.width,
            src.height,
            *src.bounds,
        )

        destination = np.full(
            (height, width),
            np.nan,
            dtype=np.float32,
        )
        source = src.read(1).astype(np.float32)

        if src.nodata is not None:
            source[source == src.nodata] = np.nan

        reproject(
            source=source,
            destination=destination,
            src_transform=src.transform,
            src_crs=src.crs,
            src_nodata=np.nan,
            dst_transform=dst_transform,
            dst_crs=target_crs,
            dst_nodata=np.nan,
            resampling=Resampling.nearest,
        )

        left = dst_transform.c
        top = dst_transform.f
        right = left + dst_transform.a * width
        bottom = top + dst_transform.e * height

    return np.ma.masked_invalid(destination), (left, right, bottom, top)


def longitude_formatter(value, position=None):
    hemisphere = "W" if value < 0 else "E"
    return f"{abs(value):.2f}°{hemisphere}"


def latitude_formatter(value, position=None):
    hemisphere = "S" if value < 0 else "N"
    return f"{abs(value):.2f}°{hemisphere}"


def draw_basin_boundary(ax):
    for feature in basin_features:
        for exterior in iter_polygon_exteriors(feature["geometry"]):
            x, y = exterior.xy
            ax.plot(
                x,
                y,
                linewidth=1.15,
                color="black",
                zorder=4,
            )


def draw_stations(ax):
    for feature in station_features:
        geometry = feature["geometry"]
        if geometry.geom_type != "Point":
            continue

        properties = feature["properties"]
        station_id = str(properties.get("station_id", ""))
        inside_basin = bool(properties.get("inside_basin", True))

        marker = "^" if inside_basin else "D"

        ax.scatter(
            [geometry.x],
            [geometry.y],
            marker=marker,
            s=46 if inside_basin else 40,
            facecolor="white",
            edgecolor="black",
            linewidth=0.9,
            zorder=6,
        )

        offset = STATION_LABEL_OFFSETS.get(station_id, (6, 6))
        ax.annotate(
            station_id,
            (geometry.x, geometry.y),
            xytext=offset,
            textcoords="offset points",
            fontsize=7.5,
            fontweight="bold",
            color="black",
            zorder=7,
            bbox=dict(
                boxstyle="round,pad=0.16",
                facecolor="white",
                edgecolor="none",
                alpha=0.82,
            ),
        )


def add_north_arrow(ax):
    ax.annotate(
        "N",
        xy=(0.94, 0.92),
        xytext=(0.94, 0.83),
        xycoords="axes fraction",
        textcoords="axes fraction",
        ha="center",
        va="center",
        fontsize=9,
        fontweight="bold",
        color="black",
        arrowprops=dict(
            arrowstyle="-|>",
            linewidth=1.0,
            color="black",
        ),
        zorder=20,
    )


def add_scale_bar(ax, length_km=10.0):
    xmin, xmax = ax.get_xlim()
    ymin, ymax = ax.get_ylim()

    lon0 = xmin + (xmax - xmin) * 0.07
    lat0 = ymin + (ymax - ymin) * 0.07

    x0, y0 = transform(
        "EPSG:4326",
        "EPSG:32618",
        [lon0],
        [lat0],
    )
    lon1, lat1 = transform(
        "EPSG:32618",
        "EPSG:4326",
        [x0[0] + length_km * 1000.0],
        [y0[0]],
    )

    lon1 = lon1[0]
    lat1 = lat1[0]

    ax.plot(
        [lon0, lon1],
        [lat0, lat1],
        linewidth=2.1,
        color="black",
        solid_capstyle="butt",
        zorder=20,
    )
    ax.text(
        (lon0 + lon1) / 2,
        max(lat0, lat1) + 0.0025,
        f"{length_km:g} km",
        ha="center",
        va="bottom",
        fontsize=7,
        color="black",
        zorder=20,
    )


def setup_map_axes(ax):
    xmin, ymin, xmax, ymax = MAP_BOUNDS
    xpad = (xmax - xmin) * 0.04
    ypad = (ymax - ymin) * 0.04

    ax.set_xlim(xmin - xpad, xmax + xpad)
    ax.set_ylim(ymin - ypad, ymax + ypad)

    mid_lat = 0.5 * (ymin + ymax)
    ax.set_aspect(1.0 / np.cos(np.deg2rad(mid_lat)))

    ax.xaxis.set_major_locator(MaxNLocator(nbins=6))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=6))
    ax.xaxis.set_major_formatter(FuncFormatter(longitude_formatter))
    ax.yaxis.set_major_formatter(FuncFormatter(latitude_formatter))

    ax.tick_params(axis="x", labelsize=7)
    ax.tick_params(axis="y", labelsize=7, labelrotation=90)
    for label in ax.get_yticklabels():
        label.set_verticalalignment("center")

    ax.grid(False)
    draw_basin_boundary(ax)
    draw_stations(ax)
    add_north_arrow(ax)
    add_scale_bar(ax)


def add_inside_legend(ax):
    handles = [
        Line2D(
            [0],
            [0],
            color="black",
            linewidth=1.15,
            label="Basin boundary",
        ),
        Line2D(
            [0],
            [0],
            marker="^",
            linestyle="None",
            markerfacecolor="white",
            markeredgecolor="black",
            markersize=6,
            label="In-basin station",
        ),
        Line2D(
            [0],
            [0],
            marker="D",
            linestyle="None",
            markerfacecolor="white",
            markeredgecolor="black",
            markersize=5,
            label="External station",
        ),
    ]

    ax.legend(
        handles=handles,
        loc="upper left",
        fontsize=6.2,
        frameon=True,
        framealpha=0.88,
        borderpad=0.35,
        labelspacing=0.25,
        handlelength=1.4,
    )


def add_inside_colorbar(fig, ax, image):
    cax = inset_axes(
        ax,
        width="3.5%",
        height="33%",
        loc="lower right",
        borderpad=0.9,
    )
    colorbar = fig.colorbar(
        image,
        cax=cax,
        extend="max",
    )
    colorbar.set_label(
        "ET (mm / 8 days)",
        fontsize=6.5,
    )
    colorbar.ax.tick_params(labelsize=6.5)

## Load fine and native MODIS rasters

In [ ]:
def find_single(pattern: str) -> Path:
    matches = sorted(WORKSPACE.glob(pattern))
    if len(matches) != 1:
        raise FileNotFoundError(f"Expected exactly one match for {pattern}; found {len(matches)}")
    return matches[0]


fine_paths = {
    date: find_single(f"current/rasters/{date}/ET_rf25_*_{date}_20m.tif")
    for date in FINAL_DATES
}
modis_paths = {
    date: find_single(f"current/rasters_modis/{date}/MODIS_ET_{date}_native.tif")
    for date in FINAL_DATES
}

fine_display = {}
fine_extents = {}
modis_display = {}
modis_extents = {}

for date in FINAL_DATES:
    fine_display[date], fine_extents[date] = reproject_raster_for_display(fine_paths[date])
    modis_display[date], modis_extents[date] = reproject_raster_for_display(modis_paths[date])

for date in FINAL_DATES:
    print(date, "fine max:", float(fine_display[date].max()), "MODIS max:", float(modis_display[date].max()))

print(f"Fixed display scale: {DISPLAY_VMIN:.0f}–{DISPLAY_VMAX:.0f} mm / 8 days")


## M01–M03 — Fine downscaled ET

In [ ]:
for index, date in enumerate(FINAL_DATES, start=1):
    fig, ax = plt.subplots(figsize=(7.5, 5.8))

    image = ax.imshow(
        fine_display[date],
        extent=fine_extents[date],
        origin="upper",
        vmin=DISPLAY_VMIN,
        vmax=DISPLAY_VMAX,
        cmap="viridis",
        interpolation="nearest",
        zorder=1,
    )

    setup_map_axes(ax)
    add_inside_legend(ax)
    add_inside_colorbar(fig, ax, image)

    ax.set_title(
        f"RF-25 downscaled evapotranspiration — {date}",
        fontsize=11,
    )
    ax.set_xlabel("Longitude", fontsize=8)
    ax.set_ylabel("Latitude", fontsize=8)

    fig.tight_layout()

    output = OUTPUT_DIR / f"M0{index}_fine_et_{date}.png"
    fig.savefig(output, dpi=300, bbox_inches="tight")
    plt.show()

    print(output)

## M04–M06 — Native MODIS ET

In [ ]:
for index, date in enumerate(FINAL_DATES, start=4):
    fig, ax = plt.subplots(figsize=(7.5, 5.8))

    image = ax.imshow(
        modis_display[date],
        extent=modis_extents[date],
        origin="upper",
        vmin=DISPLAY_VMIN,
        vmax=DISPLAY_VMAX,
        cmap="viridis",
        interpolation="nearest",
        zorder=1,
    )

    setup_map_axes(ax)
    add_inside_legend(ax)
    add_inside_colorbar(fig, ax, image)

    ax.set_title(
        f"MODIS evapotranspiration — {date}\n"
        "Native grid (nominal 500 m)",
        fontsize=11,
    )
    ax.set_xlabel("Longitude", fontsize=8)
    ax.set_ylabel("Latitude", fontsize=8)

    fig.tight_layout()

    output = OUTPUT_DIR / f"M0{index}_modis_et_{date}.png"
    fig.savefig(output, dpi=300, bbox_inches="tight")
    plt.show()

    print(output)

## M07 — Fine versus native MODIS ET for the same three periods

In [ ]:
fig, axes = plt.subplots(
    2,
    3,
    figsize=(15.3, 9.6),
    constrained_layout=True,
)

image = None

for column, date in enumerate(FINAL_DATES):
    ax = axes[0, column]
    image = ax.imshow(
        fine_display[date],
        extent=fine_extents[date],
        origin="upper",
        vmin=DISPLAY_VMIN,
        vmax=DISPLAY_VMAX,
        cmap="viridis",
        interpolation="nearest",
        zorder=1,
    )
    setup_map_axes(ax)
    ax.set_title(date, fontsize=10)
    if column == 0:
        ax.set_ylabel("Latitude\n\nRF-25 ET\n20 m grid", fontsize=8)
    ax.set_xlabel("Longitude", fontsize=7)

    ax = axes[1, column]
    image = ax.imshow(
        modis_display[date],
        extent=modis_extents[date],
        origin="upper",
        vmin=DISPLAY_VMIN,
        vmax=DISPLAY_VMAX,
        cmap="viridis",
        interpolation="nearest",
        zorder=1,
    )
    setup_map_axes(ax)
    if column == 0:
        ax.set_ylabel(
            "Latitude\n\nMODIS ET\nnative grid",
            fontsize=8,
        )
    ax.set_xlabel("Longitude", fontsize=7)

add_inside_legend(axes[0, 0])
add_inside_colorbar(fig, axes[1, -1], image)

fig.suptitle(
    "Downscaled and native MODIS evapotranspiration",
    fontsize=13,
)

output = OUTPUT_DIR / "M07_fine_vs_modis_three_periods.png"
fig.savefig(output, dpi=300, bbox_inches="tight")
plt.show()

print(output)

## Generated maps

In [ ]:
for path in sorted(OUTPUT_DIR.glob("M*.png")):
    print(path.name)